# 03 – Hypothesis tests for RQ1, RQ2 and RQ3
**QM640 Data Analytics Capstone – Binesh Balakrishnan**

Dependent variable: **SLA breach** (1 = breached). Tests at α = 0.05. Input: `data/processed/incidents.csv`. Output: `results/rq1_rq3.json`.

In [1]:
import os, subprocess
repo = '/content/QM640_data_analytics_capstone'
if not os.path.exists(repo):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/binesh29/QM640_data_analytics_capstone.git', repo], check=True)
os.chdir(os.path.join(repo, 'notebooks'))
os.makedirs('../figures', exist_ok=True); os.makedirs('../results', exist_ok=True)

In [2]:
os.makedirs('../src', exist_ok=True)
with open('../src/plot_style.py', 'w') as f:
    f.write('''import matplotlib as mpl
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
def apply():
    mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.edgecolor': INK2,
        'axes.labelcolor': INK, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
        'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
        'grid.linewidth': 0.8, 'axes.axisbelow': True, 'xtick.color': INK2, 'ytick.color': INK2,
        'legend.frameon': False, 'figure.dpi': 110, 'savefig.dpi': 200, 'savefig.bbox': 'tight', 'lines.linewidth': 2})
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2 * n)) / d
    h = z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5) / d
    return c - h, c + h
''')
print(os.listdir('../src'))

['sample_size.py', 'plot_style.py']


In [3]:
import json, itertools
import numpy as np, pandas as pd
from scipy.stats import chi2_contingency, norm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable, Table2x2
inc = pd.read_csv('../data/processed/incidents.csv')
# controls: priority (4 levels) and category, with categories under 30 incidents grouped as Other
cnt = inc['category'].fillna('Missing').value_counts()
inc['cat_grp'] = inc['category'].fillna('Missing').where(lambda s: s.map(cnt) >= 30, 'Other')
inc['knowledge'] = inc['knowledge'].astype(int)
print(inc['cat_grp'].nunique(), 'category groups')
R = {'category_groups': int(inc['cat_grp'].nunique())}

31 category groups


## RQ1 – Priority × SLA breach (χ² test of independence, 4 × 2)

In [4]:
t = pd.crosstab(inc['priority'], inc['sla_breach'])
chi2, p, dof, exp = chi2_contingency(t, correction=False)
n = t.values.sum(); V = np.sqrt(chi2 / (n * (min(t.shape) - 1)))
# adjusted standardised residuals
rs = t.sum(1).values[:, None] / n; cs = t.sum(0).values[None, :] / n
adj = (t.values - exp) / np.sqrt(exp * (1 - rs) * (1 - cs))
R['rq1'] = {'chi2': round(chi2, 1), 'df': int(dof), 'p': float(p), 'cramers_v': round(V, 3), 'min_expected': round(exp.min(), 1),
            'adj_resid_breach': dict(zip(t.index, adj[:, 1].round(1)))}
pairs = []
for a, b in itertools.combinations(t.index, 2):
    sub = t.loc[[a, b]]; c2, pp, _, _ = chi2_contingency(sub, correction=False)
    pairs.append([a, b, round(sub.loc[a, 1] / sub.loc[a].sum() * 100, 1), round(sub.loc[b, 1] / sub.loc[b].sum() * 100, 1), round(c2, 1), pp])
pw = pd.DataFrame(pairs, columns=['A', 'B', 'breach_A_%', 'breach_B_%', 'chi2', 'p'])
pw['p_holm'] = multipletests(pw['p'], method='holm')[1]; pw['reject'] = pw['p_holm'] < 0.05
R['rq1']['pairwise'] = pw.to_dict('records')
print(R['rq1']); pw

{'chi2': np.float64(1293.1), 'df': 3, 'p': 4.544927427470316e-280, 'cramers_v': np.float64(0.228), 'min_expected': np.float64(98.8), 'adj_resid_breach': {'1 - Critical': np.float64(21.1), '2 - High': np.float64(26.6), '3 - Moderate': np.float64(-14.8), '4 - Low': np.float64(-12.1)}, 'pairwise': [{'A': '1 - Critical', 'B': '2 - High', 'breach_A_%': 98.1, 'breach_B_%': 99.5, 'chi2': 2.9, 'p': 0.08596937188122478, 'p_holm': 0.08596937188122478, 'reject': False}, {'A': '1 - Critical', 'B': '3 - Moderate', 'breach_A_%': 98.1, 'breach_B_%': 35.5, 'chi2': 454.3, 'p': 8.199252147883869e-101, 'p_holm': 2.4597756443651606e-100, 'reject': True}, {'A': '1 - Critical', 'B': '4 - Low', 'breach_A_%': 98.1, 'breach_B_%': 15.9, 'chi2': 580.0, 'p': 3.7908597657097647e-128, 'p_holm': 1.5163439062839059e-127, 'reject': True}, {'A': '2 - High', 'B': '3 - Moderate', 'breach_A_%': 99.5, 'breach_B_%': 35.5, 'chi2': 709.4, 'p': 2.746755129792139e-156, 'p_holm': 1.3733775648960693e-155, 'reject': True}, {'A': '

,A,B,breach_A_%,breach_B_%,chi2,p,p_holm,reject
0,1 - Critical,2 - High,98.1,99.5,2.9,8.596937e-02,8.596937e-02,False
1,1 - Critical,3 - Moderate,98.1,35.5,454.3,8.199252e-101,2.459776e-100,True
2,1 - Critical,4 - Low,98.1,15.9,580.0,3.790860e-128,1.516344e-127,True
3,2 - High,3 - Moderate,99.5,35.5,709.4,2.746755e-156,1.373378e-155,True
4,2 - High,4 - Low,99.5,15.9,755.5,2.515354e-166,1.509212e-165,True
5,3 - Moderate,4 - Low,35.5,15.9,126.4,2.521565e-29,5.043130e-29,True


## RQ2 – Reassignment × SLA breach

In [5]:
t2 = pd.crosstab(inc['reassigned'], inc['sla_breach'])
c2, p2, _, _ = chi2_contingency(t2, correction=False)
tb = Table2x2(t2.loc[[1, 0], [1, 0]].values)
R['rq2'] = {'chi2': round(c2, 1), 'p': float(p2), 'phi': round(np.sqrt(c2 / t2.values.sum()), 3),
            'or_unadj': round(tb.oddsratio, 2), 'or_unadj_ci': [round(x, 2) for x in tb.oddsratio_confint()]}
m2 = smf.logit('sla_breach ~ reassignment_count + C(priority) + C(cat_grp)', data=inc).fit(disp=0, maxiter=200)
b, se = m2.params['reassignment_count'], m2.bse['reassignment_count']
R['rq2'].update({'beta': round(b, 4), 'se': round(se, 4), 'z': round(b / se, 2), 'p_wald': float(m2.pvalues['reassignment_count']),
                 'or_adj': round(np.exp(b), 3), 'or_adj_ci': [round(np.exp(b - 1.96 * se), 3), round(np.exp(b + 1.96 * se), 3)],
                 'mcfadden_r2': round(m2.prsquared, 3), 'n': int(m2.nobs), 'converged': bool(m2.mle_retvals['converged'])})
# linearity check: reassignment as categories 0, 1, 2, 3+
inc['reassign_cat'] = pd.cut(inc['reassignment_count'], [-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+'])
m2c = smf.logit('sla_breach ~ C(reassign_cat) + C(priority) + C(cat_grp)', data=inc).fit(disp=0, maxiter=200)
R['rq2']['or_by_level'] = {k.split('T.')[1].rstrip(']'): round(np.exp(v), 2) for k, v in m2c.params.items() if 'reassign_cat' in k}
# variance inflation of reassignment_count on the controls
aux = smf.ols('reassignment_count ~ C(priority) + C(cat_grp)', data=inc).fit()
R['rq2']['vif'] = round(1 / (1 - aux.rsquared), 3)
print(R['rq2'])

{'chi2': np.float64(2868.4), 'p': 0.0, 'phi': np.float64(0.339), 'or_unadj': np.float64(4.33), 'or_unadj_ci': [np.float64(4.1), np.float64(4.58)], 'beta': np.float64(0.667), 'se': np.float64(0.0138), 'z': np.float64(48.46), 'p_wald': 0.0, 'or_adj': np.float64(1.948), 'or_adj_ci': [np.float64(1.896), np.float64(2.002)], 'mcfadden_r2': np.float64(0.217), 'n': 24918, 'converged': True, 'or_by_level': {'1': np.float64(2.88), '2': np.float64(5.89), '3+': np.float64(12.78)}, 'vif': np.float64(1.049)}


## RQ3 – Knowledge-base use × SLA breach

In [6]:
t3 = pd.crosstab(inc['knowledge'], inc['sla_breach'])
c3, p3, _, _ = chi2_contingency(t3, correction=False)
tb3 = Table2x2(t3.loc[[1, 0], [1, 0]].values)
R['rq3'] = {'table': t3.values.tolist(), 'chi2': round(c3, 1), 'p': float(p3), 'phi': round(np.sqrt(c3 / t3.values.sum()), 3),
            'or_unadj': round(tb3.oddsratio, 2), 'or_unadj_ci': [round(x, 2) for x in tb3.oddsratio_confint()]}
m3 = smf.logit('sla_breach ~ knowledge + C(priority) + C(cat_grp)', data=inc).fit(disp=0, maxiter=200)
b, se = m3.params['knowledge'], m3.bse['knowledge']
R['rq3'].update({'beta': round(b, 4), 'se': round(se, 4), 'z': round(b / se, 2), 'p_wald': float(m3.pvalues['knowledge']),
                 'or_adj': round(np.exp(b), 2), 'or_adj_ci': [round(np.exp(b - 1.96 * se), 2), round(np.exp(b + 1.96 * se), 2)],
                 'mcfadden_r2': round(m3.prsquared, 3)})
aux3 = smf.ols('knowledge ~ C(priority) + C(cat_grp)', data=inc).fit(); R['rq3']['vif'] = round(1 / (1 - aux3.rsquared), 3)
# Mantel–Haenszel across the four priority strata
strata = [pd.crosstab(g['knowledge'], g['sla_breach']).reindex(index=[1, 0], columns=[1, 0], fill_value=0).values
          for _, g in inc.groupby('priority')]
st = StratifiedTable(strata)
R['rq3'].update({'mh_or': round(st.oddsratio_pooled, 2), 'mh_ci': [round(x, 2) for x in st.oddsratio_pooled_confint()],
                 'mh_p': float(st.test_null_odds().pvalue), 'breslow_day_p': float(st.test_equal_odds().pvalue),
                 'stratum_or': {p: round(Table2x2(s + 0.5).oddsratio, 2) for p, s in zip(sorted(inc['priority'].unique()), strata)}})
# sensitivity: also adjust for reassignment
m3s = smf.logit('sla_breach ~ knowledge + reassignment_count + C(priority) + C(cat_grp)', data=inc).fit(disp=0, maxiter=200)
R['rq3']['or_adj_plus_reassign'] = round(np.exp(m3s.params['knowledge']), 2)
print(R['rq3'])

{'table': [[14853, 6514], [950, 2601]], 'chi2': np.float64(2399.9), 'p': 0.0, 'phi': np.float64(0.31), 'or_unadj': np.float64(6.24), 'or_unadj_ci': [np.float64(5.76), np.float64(6.76)], 'beta': np.float64(1.8906), 'se': np.float64(0.0461), 'z': np.float64(41.01), 'p_wald': 0.0, 'or_adj': np.float64(6.62), 'or_adj_ci': [np.float64(6.05), np.float64(7.25)], 'mcfadden_r2': np.float64(0.171), 'vif': np.float64(1.176), 'mh_or': np.float64(7.26), 'mh_ci': [np.float64(6.68), np.float64(7.89)], 'mh_p': 0.0, 'breslow_day_p': 0.00598235348039855, 'stratum_or': {'1 - Critical': np.float64(0.42), '2 - High': np.float64(1.21), '3 - Moderate': np.float64(7.18), '4 - Low': np.float64(11.06)}, 'or_adj_plus_reassign': np.float64(6.6)}


## Sensitivity – structural break in March 2016 (control for opening week)

In [7]:
inc['opened_at'] = pd.to_datetime(inc['opened_at'])
inc['week'] = inc['opened_at'].dt.to_period('W').astype(str)
wc = inc['week'].value_counts(); inc['wk'] = inc['week'].where(inc['week'].map(wc) >= 100, 'other')
pre = inc['opened_at'] < '2016-03-21'
R['knowledge_share_pre_0321'] = round(inc.loc[pre, 'knowledge'].mean(), 3); R['knowledge_share_post_0321'] = round(inc.loc[~pre, 'knowledge'].mean(), 3)
R['breach_pre_0321'] = round(inc.loc[pre, 'sla_breach'].mean(), 3); R['breach_post_0321'] = round(inc.loc[~pre, 'sla_breach'].mean(), 3)
R['n_pre_0321'] = int(pre.sum())
for key, var in [('rq2', 'reassignment_count'), ('rq3', 'knowledge')]:
    mw = smf.logit(f'sla_breach ~ {var} + C(priority) + C(cat_grp) + C(wk)', data=inc).fit(disp=0, maxiter=300)
    b, se = mw.params[var], mw.bse[var]
    R[key]['week_adjusted'] = {'or': round(np.exp(b), 3), 'ci': [round(np.exp(b - 1.96 * se), 3), round(np.exp(b + 1.96 * se), 3)], 'p': float(mw.pvalues[var])}
print(R['rq2']['week_adjusted'], R['rq3']['week_adjusted'], R['knowledge_share_pre_0321'], R['knowledge_share_post_0321'], R['breach_pre_0321'], R['breach_post_0321'])

{'or': np.float64(1.931), 'ci': [np.float64(1.876), np.float64(1.988)], 'p': 0.0} {'or': np.float64(0.614), 'ci': [np.float64(0.439), np.float64(0.86)], 'p': 0.004485621737751595} 0.586 0.005 0.731 0.252


In [8]:
json.dump(R, open('../results/rq1_rq3.json', 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o)); print('saved')

saved
